DATA PREPROCESSING

In [23]:
# STEP 1: LOAD DATA
import pandas as pd

df = pd.read_csv(
    "training.1600000.processed.noemoticon.csv",
    encoding= "latin-1", # It is used to prevent errors when encountering non-UTF-8 characters. eg : emoji, symbols
    usecols= ['polarity of tweet','text of the tweet'] #here the columns we are using idex of [0, 5]
    # polarity -> sentiment analysis
    # text -  text content
)
print(df.shape)
print(df.head())

(1048572, 2)
   polarity of tweet                                  text of the tweet
0                  0  is upset that he can't update his Facebook by ...
1                  0  @Kenichan I dived many times for the ball. Man...
2                  0    my whole body feels itchy and like its on fire 
3                  0  @nationwideclass no, it's not behaving at all....
4                  0                      @Kwesidei not the whole crew 


In [24]:
# STEP 2: Sampling 10,000 Tweets

df = df.sample(n = 10000 ,random_state= 42) #sample() randomly selects rows from the dataset.

# Reset index after sampling 
df = df.reset_index(drop = True)

print(df.shape)
print(df.head())

(10000, 2)
   polarity of tweet                                  text of the tweet
0                  4  just came bak from dancing with my NEEWWWW cd ...
1                  0  Post office, n other runnin around to do...gee...
2                  0  @SabrinaL OOOOOOH! This song....I hope he know...
3                  0  I wish iwasnt here im think'n &amp; being arou...
4                  4  http://twitpic.com/3kyv5 - All i do is twitter...


In [25]:
# STEP 3: Data Understanding 

print((df['polarity of tweet'].unique())) #it has 0, 4 => pos, 0 => neg
print(df['polarity of tweet'].value_counts()) #it will count the values of 0 and 4

[4 0]
polarity of tweet
0    7658
4    2342
Name: count, dtype: int64


In [26]:
# STEP 4: LABEL MAPPING

df['polarity of tweet'] = df['polarity of tweet'].map({
    0 : 0 , #neg
    4 : 1   #pos
}) # mapping the polarity, training the models using the frameworks like 'Pytorch' and 'Tensorflow', use continuous class labels starting from 0.

print(df['polarity of tweet'].value_counts())
print(df.shape)

polarity of tweet
0    7658
1    2342
Name: count, dtype: int64
(10000, 2)


NLP PREPROCESSING

In [27]:
# STEP 5: Text Cleaning
# We are removing the URL, Space, emojis , unwanted chars
# If we don't clean, Model will treat them as words, which adds noise. to the RNN, LSTM, GRU.

import re #re -> Regural Expression => advanced string manipulation, pattern matching, searching, and splitting using specialized syntax.

def clean_text(text):

    # Change the Text into lowercase
    text = text.lower() 

    # Remove Integer values 
    text = re.sub(r'\d+', ' ',text) # \d → digit (0-9) , withou "r" python will consider the \ as an variable
 
    # Remove URL 
    # r → raw string (so python doesn't treat \ as escape).  http → find text starting with "http"
    # \S → any non-space character.  + → one or more
    # http\S+ → matches full URL like http://abc.com
    text = re.sub(r'http\S+',' ', text)


    # Remove Mentions
    # @ → literal @ symbol . \w → word characters (a-z, A-Z, 0-9, _).  + → one or more
    # @\w+ → matches @john, @user123
    text = re.sub(r'@\w+', ' ', text)


    #Remove special characters
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)  #[^a-zA-Z\s] -> emojis, punctuation

    
    # Remove Space
    text = re.sub(r'\s+', ' ', text)


    # strip() removes spaces at beginning and end
    text = text.strip()

    return text

# Apply clean_text to all the columns
df['text of the tweet'] = df['text of the tweet'].apply(clean_text) #apply -> used to apply the function into all columns 

# Calculate the Text length 
# why we call this, bcoz the DL models like TensorFlow, GRU these all need fixed input size, so we are calculating the text.
df['text_length'] = df['text of the tweet'].apply(lambda x: len(x.split())) #lambda x:len(x.split()) -> lambda => small temp function, 1st sentences, atha split panitu athoda len ah use panrom

print(df['text of the tweet'].head())
print(df['text_length'].head())

# http\S+   # URLs
# @\w+      # mentions
# \d+       # numbers
# \s+       # extra spaces
# lower()   # lowercase

0    just came bak from dancing with my neewwww cd ...
1    post office n other runnin around to do geesh ...
2    ooooooh this song i hope he knows about this o...
3    i wish iwasnt here im think n amp being around...
4          kyv all i do is twitter according to thalia
Name: text of the tweet, dtype: object
0    13
1    23
2    16
3    18
4     9
Name: text_length, dtype: int64


In [28]:
print(df['text_length'].describe())

count    10000.000000
mean        13.366700
std          7.309116
min          0.000000
25%          7.000000
50%         12.000000
75%         19.000000
max         35.000000
Name: text_length, dtype: float64


In [29]:
# import re

# text = "I Love vs code version 2"
# clean_test = re.sub(r'\d+',' ',text)
# print(clean_test)

#Tokenization

Tokenization → words become numbers
Embedding → numbers become meaning vectors
DL model → learns patterns from vectors

In [30]:
# STEP 6 : Tokenization 
# Which is used to convert the words into numbers 


# Importing Token
from tensorflow.keras.preprocessing.text import Tokenizer

# creating tokens
tokens = Tokenizer(num_words = 5000) #num_words -> top frequent words, without this it will occur overfitting, Noise, Training time will increase

# Learn vocalboary
tokens.fit_on_texts(df['text of the tweet']) #it scans entire dataset and built vocalbulary. EG: I love this , I - 1, love - 2, this - 3.

# Convert Text -> Sequences(text -> sequnce of numbers)
sequence = tokens.texts_to_sequences(df['text of the tweet'])

# Store Sequence 
df['sequence'] = sequence #sequence , which is list of numbers which is vocal numbers.

# Check the Sequence
print(df[['text of the tweet', 'sequence']].head())
 
# checking the vocal sixe
vocal_size = len(tokens.word_index)+1
print("The size of Vocal: ",vocal_size) #it contain the unique words in the dataset 

                                   text of the tweet                                           sequence
0  just came bak from dancing with my neewwww cd ...  [23, 436, 1262, 52, 1023, 28, 5, 825, 133, 56,...
1  post office n other runnin around to do geesh ...  [475, 437, 187, 268, 298, 2, 46, 30, 1, 327, 3...
2  ooooooh this song i hope he knows about this o...  [27, 342, 1, 117, 75, 1024, 63, 27, 1372, 75, ...
3  i wish iwasnt here im think n amp being around...  [1, 92, 79, 56, 73, 187, 65, 140, 298, 102, 33...
4        kyv all i do is twitter according to thalia                       [35, 1, 46, 9, 106, 1891, 2]
The size of Vocal:  13320


In [31]:
# STEP 7: Sequence Padding 
# Why this process? -> The sequence may have different length, we are using RNN / LSTM / GRU need same length input.That why we are using padding.
# eg : [3,2,1] , padding max_len = 5 => [3,2,1,0,0]. after it all the sequence has same lenght.

# Importing Sequence
from tensorflow.keras.preprocessing.sequence import pad_sequences

# setting the max length
max_len = 35

pad_x = pad_sequences(df['sequence'], maxlen = max_len, padding = 'post') #this will apply for each line in the dataset 
# maxlen(max len -> parameter) = 35, padding = 'post' => add 0 at the end. 

print(pad_x[0])
print(pad_x.shape) #10000, 35

[  23  436 1262   52 1023   28    5  825  133   56 1023  100    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0]
(10000, 35)


In [34]:
# STEP 8: TRAIN- TEST- SPLIT 

from sklearn.model_selection import train_test_split

x_train, x_test, y_train , y_test = train_test_split(
    pad_x, #Xlabel
    df['polarity of tweet'], #Ylabel
    test_size= 0.2,
    random_state= 42
)

print(x_train.shape)
print(x_test.shape)
print(y_train.shape)
print(y_test.shape)

(8000, 35)
(2000, 35)
(8000,)
(2000,)
